# Copy-vs-use control — Qwen3-0.6B

Run in Google Colab: **Runtime → Change runtime type → T4 GPU**, then **Run all**.
The notebook is self-contained; no repository upload or token is required.
It downloads Qwen3-0.6B and evaluates 30 fixed cases in three conditions (90 forward passes).

The wrong-working prompt separates using the planted intermediate result, copying it,
and recomputing from the question by their distinct first digits. Cases are selected
using arithmetic constraints and seed 42 before model evaluation. Correct-working and
no-working prompts are paired baselines. All conditions retain the same three target
labels, defined by the wrong-working hypotheses.

We measure unconditional next-token probabilities, not full-answer correctness.
Global top tokens outside the three candidate digits count as `other`. Probabilities
are not renormalized over digits. This uses original Hugging Face weights; the older
circuit notebook used a replacement model with transcoders, so compare cautiously.
No claim about 1.7B or self-generated reasoning follows from this control.


In [ ]:
%pip -q install "transformers>=4.51.0,<5" torch

The script below is embedded from `code/src/run_copy_vs_use.py`. Regenerate this cell if the source script changes.

In [ ]:
from pathlib import Path
script_path = Path("/content/run_copy_vs_use.py")
script_path.write_text('"""First-token copy-vs-use control. No model dependencies needed for --dry-run.\n\nThe Colab notebook embeds this script; regenerate its script cell after edits.\n"""\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport random\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nCATEGORIES = ("use", "copy", "true")\nCONDITIONS = ("wrong", "correct", "none")\n\n\ndef make_cases():\n    """Fixed seed and arithmetic-only selection; never select on model outputs."""\n    candidates = []\n    for a in range(11, 40):\n        for b in range(11, 40):\n            for multiplier in (2, 3, 4):\n                for wrong in (21, 32, 43, 54, 65, 76, 87, 98):\n                    targets = {"use": wrong * multiplier, "copy": wrong,\n                               "true": (a + b) * multiplier}\n                    if len({str(v)[0] for v in targets.values()}) == 3:\n                        candidates.append(dict(a=a, b=b, multiplier=multiplier,\n                                               wrong=wrong, targets=targets))\n    random.Random(42).shuffle(candidates)\n    cases, seen = [], set()\n    for case in candidates:\n        key = (case["a"], case["b"], case["multiplier"])\n        if key in seen:\n            continue\n        seen.add(key)\n        cases.append(dict(id=len(cases), **case))\n        if len(cases) == 30:\n            return cases\n    raise RuntimeError("Too few eligible cases")\n\n\ndef make_prompt(case, condition):\n    prompt = f"Question: ({case[\'a\']} + {case[\'b\']}) * {case[\'multiplier\']} = ?\\n"\n    if condition != "none":\n        value = case["wrong"] if condition == "wrong" else case["a"] + case["b"]\n        prompt += f"Working: {case[\'a\']} + {case[\'b\']} = {value}.\\n"\n    return prompt + "Final answer: "\n\n\ndef manifest():\n    return [dict(**case, condition=condition, prompt=make_prompt(case, condition))\n            for case in make_cases() for condition in CONDITIONS]\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()\n\n\ndef summarize(rows):\n    expected = manifest()\n    if len(rows) != len(expected):\n        raise ValueError("Need all 90 records before producing a paper summary")\n    for row, spec in zip(rows, expected):\n        if any(row.get(k) != v for k, v in spec.items()):\n            raise ValueError("Raw records do not match the fixed prompt manifest")\n        probs = row["digit_probabilities"]\n        if set(probs) != set("0123456789") or any(not 0 <= p <= 1 for p in probs.values()):\n            raise ValueError("Invalid digit probabilities")\n        if sum(probs.values()) > 1 + 1e-5:\n            raise ValueError("Digit probabilities exceed vocabulary probability mass")\n    result = {"n_cases": 30, "n_forward_passes": len(rows), "conditions": {}}\n    for condition in CONDITIONS:\n        subset = [r for r in rows if r["condition"] == condition]\n        data = {}\n        # In baselines these labels still refer to the WRONG-working hypotheses.\n        for category in CATEGORIES:\n            values = [r["digit_probabilities"][str(r["targets"][category])[0]] for r in subset]\n            data[category] = {"mean_p": sum(values) / len(values), "min_p": min(values),\n                              "max_p": max(values)}\n        winners = Counter()\n        for row in subset:\n            winner = next((c for c in CATEGORIES if row["top_token_id"] ==\n                           row["digit_token_ids"][str(row["targets"][c])[0]]), "other")\n            winners[winner] += 1\n        data["global_top_token_counts"] = {c: winners[c] for c in (*CATEGORIES, "other")}\n        result["conditions"][condition] = data\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--dry-run", action="store_true")\n    parser.add_argument("--model", default="Qwen/Qwen3-0.6B")\n    parser.add_argument("--revision", default="main")\n    parser.add_argument("--output", type=Path, default=Path("outputs/raw/copy_vs_use.jsonl"))\n    parser.add_argument("--summarize", type=Path, help="Analyze a completed raw file without loading a model")\n    parser.add_argument("--summary", type=Path, default=Path("outputs/clean/copy_vs_use_summary.json"))\n    args = parser.parse_args()\n    specs = manifest()\n    if args.dry_run:\n        print(json.dumps({"status": "DESIGN ONLY; no model results", "sha256": digest(specs),\n                          "records": specs}, indent=2))\n        return\n    if args.summarize:\n        rows = [json.loads(line) for line in args.summarize.read_text().splitlines()]\n        result = summarize(rows)\n        result["raw_sha256"] = hashlib.sha256(args.summarize.read_bytes()).hexdigest()\n        args.summary.parent.mkdir(parents=True, exist_ok=True)\n        args.summary.write_text(json.dumps(result, indent=2) + "\\n")\n        print(json.dumps(result, indent=2))\n        return\n\n    import torch\n    import transformers\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    config_path = args.output.with_suffix(".config.json")\n    if args.output.exists() or config_path.exists():\n        raise FileExistsError("Preserve raw outputs: choose a new --output filename")\n    torch.manual_seed(42)\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    dtype = torch.float16 if device == "cuda" else torch.float32\n    model = AutoModelForCausalLM.from_pretrained(args.model, revision=args.revision,\n                                                torch_dtype=dtype).to(device).eval()\n    revision = model.config._commit_hash\n    if not revision:\n        raise RuntimeError("Could not resolve the model revision for provenance")\n    tokenizer = AutoTokenizer.from_pretrained(args.model, revision=revision)\n    digit_ids = {}\n    for digit in "0123456789":\n        ids = tokenizer.encode(digit, add_special_tokens=False)\n        if len(ids) != 1:\n            raise ValueError(f"Digit {digit} is not a single token: {ids}")\n        digit_ids[digit] = ids[0]\n    if len(set(digit_ids.values())) != 10:\n        raise ValueError("Digit token IDs are not distinct")\n    config = dict(model=args.model, requested_revision=args.revision, resolved_revision=revision,\n                  device=device, dtype=str(dtype), torch=torch.__version__,\n                  transformers=transformers.__version__, python=platform.python_version(),\n                  timestamp_utc=datetime.now(timezone.utc).isoformat(), seed=42,\n                  prompt_manifest_sha256=digest(specs), digit_token_ids=digit_ids,\n                  prompt_format="plain prefix; no chat template; add_special_tokens=False",\n                  model_kind="original Hugging Face weights; no replacement transcoders",\n                  metric="unconditional next-token probabilities over full vocabulary")\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    config_path.write_text(json.dumps(config, indent=2) + "\\n")\n    rows = []\n    with args.output.open("x") as handle, torch.inference_mode():\n        for spec in specs:\n            inputs = tokenizer(spec["prompt"], return_tensors="pt", add_special_tokens=False).to(device)\n            logits = model(**inputs).logits[0, -1].float()\n            probs = torch.softmax(logits, dim=-1)\n            top = int(torch.argmax(logits).item())\n            row = dict(**spec, digit_probabilities={d: probs[i].item() for d, i in digit_ids.items()},\n                       digit_token_ids=digit_ids, top_token_id=top, top_token=tokenizer.decode([top]),\n                       n_prompt_tokens=int(inputs["input_ids"].shape[1]))\n            handle.write(json.dumps(row) + "\\n")\n            handle.flush()\n            rows.append(row)\n            print(f"{len(rows)}/90: case {spec[\'id\']} {spec[\'condition\']}, top={row[\'top_token\']!r}")\n    result = summarize(rows)\n    result["raw_sha256"] = hashlib.sha256(args.output.read_bytes()).hexdigest()\n    args.summary.parent.mkdir(parents=True, exist_ok=True)\n    args.summary.write_text(json.dumps(result, indent=2) + "\\n")\n    print(json.dumps(result, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n')


## Inspect the design before running the model
This file contains prompts only; it is not a results file.

In [ ]:
!python /content/run_copy_vs_use.py --dry-run > /content/copy_vs_use_design.json
import json
design = json.loads(Path("/content/copy_vs_use_design.json").read_text())
print(design["status"], design["sha256"])
print(json.dumps(design["records"][:3], indent=2))

## Run the model
Existing raw files are protected from overwrite. For an intentional rerun choose new filenames or start a fresh runtime.

In [ ]:
import subprocess
subprocess.run(["python", "/content/run_copy_vs_use.py", "--output", "/content/copy_vs_use_results/copy_vs_use.jsonl", "--summary", "/content/copy_vs_use_results/copy_vs_use_summary.json"], check=True)

## Download and commit the results
Run this only after the model cell succeeds. The ZIP includes raw records, resolved model revision, package versions, summary, and the pre-run design.

Copy `copy_vs_use.jsonl` and `copy_vs_use.config.json` to `outputs/raw/`; put the summary and design in `outputs/clean/`. Check the wrong-condition means and global-top-token counts before updating Table IV. Do not substitute baselines for the wrong-condition results. Commit by **Wednesday 21 October, 9pm Dublin time**. Table IV stays TBD until actual results exist.

In [ ]:
import shutil
from google.colab import files
summary = Path("/content/copy_vs_use_results/copy_vs_use_summary.json")
assert summary.exists(), "Run the experiment successfully first"
shutil.copy("/content/copy_vs_use_design.json", summary.parent)
archive = shutil.make_archive("/content/copy_vs_use_results", "zip", summary.parent)
files.download(archive)